In [1]:
import sys
sys.path.insert(0, "/workspace/app")
import pandas as pd
import numpy as np
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import GridSearchCV, LeaveOneOut
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import fbeta_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, recall_score, precision_score, f1_score, average_precision_score, precision_recall_fscore_support, roc_auc_score
import math

In [2]:
df = pd.read_excel("/workspace/app/planilhas/CBMS/df_all_Horizon2.xlsx")
col_remove = ['Age', "Patient_Gender", 'Education', 'Y_Zscore_TO_TOT_Alexithymia', 'Questionnary5_HQ_25_T0_soc_Score', 
              'Questionnary5_HQ_25_T0_iso_Score', 'Questionnary5_HQ_25_T0_sup_Score', 'Mean_Gap_Days', 'Std_Gap_Days', 'Y_Depression_T0',
              "Embedding_npy_list", "Embedding_npy", "y_next"]
df = df.drop(columns=col_remove, errors="ignore")

In [3]:
df = df[df["Patient_ID"] != 2].copy()

Baseline MLP

In [4]:
class_param_grid_mlp = {
    'hidden_layer_sizes': [(100,), (50, 50), (100, 50)],
    'activation': ['relu'],
    'solver': ['adam'],
    'learning_rate': ['constant', 'adaptive'],
    'alpha': [0.0001, 0.001],  # Regularization
    'early_stopping': [True]}

In [5]:
def class_lopo(patient, df, patient_column, target_column, model, param_grid):
    train_df = df[df[patient_column] != patient]
    test_df  = df[df[patient_column] == patient]
    sessions = test_df["Session"].astype(int).tolist()
    X_train = train_df.drop(columns=['Session', target_column])
    y_train = train_df[target_column].values
    X_test  = test_df.drop(columns=['Session', target_column])
    y_test  = test_df[target_column].values
    unique_train_classes = sorted(np.unique(y_train))
    if len(unique_train_classes) < 2:
        return None  
    labels_all = sorted(set(np.unique(y_train)) | set(np.unique(y_test)))
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled  = scaler.transform(X_test)
    try:
        grid_search = GridSearchCV(
            estimator=model,
            param_grid=param_grid,
            scoring='f1_macro',
            cv=5,
            verbose=0,
            n_jobs=-1)
        sw = compute_sample_weight(class_weight="balanced", y=y_train)
        grid_search.fit(X_train_scaled, y_train, sample_weight=sw)
        best_model = grid_search.best_estimator_

        y_prob_train = best_model.predict_proba(X_train_scaled)[:, 1]
        taus = np.linspace(0.15, 0.95, 19)
        f2s = [fbeta_score(y_train, (y_prob_train >= t).astype(int), beta=2, zero_division=0) for t in taus]
        tau = float(taus[int(np.argmax(f2s))])
        tau = max(tau, float(np.quantile(y_prob_train, 0.90)))

        y_prob = best_model.predict_proba(X_test_scaled)[:, 1]
        y_pred = (y_prob >= tau).astype(int)

        bin_labels = [0, 1] if set(labels_all) == {0, 1} else labels_all
        rec_macro = recall_score(y_test, y_pred, labels=bin_labels, average='macro', zero_division=0)
        accuracy= np.mean(y_test == y_pred)
        rec_pos = recall_score(y_test, y_pred, average='binary', pos_label=1, zero_division=0)
        spec = recall_score(y_test, y_pred, average='binary', pos_label=0, zero_division=0)
        bal_acc = (rec_pos + spec) / 2
        prec    = precision_score(y_test, y_pred, zero_division=0, pos_label=1)
        f1      = f1_score(y_test, y_pred, zero_division=0, pos_label=1)
        out = {
            "Model": type(best_model).__name__,
            "Patient_ID": patient,
            "Session": sessions,
            "tau": tau,
            "Best_Params":  grid_search.best_params_,
            "Recall":                float(rec_macro),  # Macro-per-fold (M)
            "Accuracy":              float(accuracy),
            "BalancedAccuracy":     float(bal_acc),
            "Recall_Pos":            float(rec_pos),    # Positive-class (P)
            "Specificity":           float(spec),
            "F1_Score":              float(f1),         # P
            "Precision":             float(prec),       # P
            "y_true":  y_test.tolist(),
            "y_pred":  y_pred.tolist(),
            "y_prob": y_prob.tolist()}
        return out
    except ValueError:
        return None

In [6]:
summary_results = []
all_results = []
unique_patients = df['Patient_ID'].unique()
target_column = 'y_h2'
results_mlp = [class_lopo(patient, df, 'Patient_ID', target_column, MLPClassifier(random_state=42), class_param_grid_mlp)
    for patient in unique_patients]
all_results.extend(results_mlp)
df_current = pd.DataFrame(results_mlp)
summary_results.append({
    "Recall_Macro_Mean": df_current["Recall"].mean(),
    "Recall_Macro_Std": df_current["Recall"].std(),
    "Accuracy_Mean": df_current["Accuracy"].mean(),
    "Accuracy_Std": df_current["Accuracy"].std(),
    "Balanced_Accuracy_Mean": df_current["BalancedAccuracy"].mean(),
    "Balanced_Accuracy_Std": df_current["BalancedAccuracy"].std(),
    "Recall_Pos_Mean": df_current["Recall_Pos"].mean(),
    "Spec_Mean": df_current["Specificity"].mean(),
    "Precision_Pos_Mean": df_current["Precision"].mean(),
    "F1_Pos_Mean": df_current["F1_Score"].mean(),
    "Tau_Mean": df_current["tau"].mean()
    })
df_results_mlp = pd.DataFrame(all_results)
df_summary_mlp = pd.DataFrame(summary_results)

In [7]:
df_results_mlp.to_excel('/workspace/app/planilhas/CBMS/MLP.xlsx', index=False)
df_summary_mlp.to_excel('/workspace/app/planilhas/CBMS/MLP_SUMMARY.xlsx', index=False)